# Muon Optimizer Development

This notebook is for developing and testing the Muon optimizer integration with the actual metamer generation code.

In [1]:
import sys
import os

# Add the project root to the path so we can import our modules
project_root = os.path.abspath('')
sys.path.insert(0, project_root)

import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models, transforms
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import pickle
import csv

print(f"Project root: {project_root}")
print(f"Python path includes project root: {project_root in sys.path}")

Project root: /orcd/data/jhm/001/om2/rphess/projects/github.com/model_metamers_pytorch
Python path includes project root: True


In [2]:
# Test Muon import
try:
    from muon import Muon
    print("✓ Muon imported successfully")
    MUON_AVAILABLE = True
except ImportError as e:
    print(f"✗ Muon import failed: {e}")
    print("Will use SGD fallback")
    MUON_AVAILABLE = False

# Test basic Muon functionality if available
if MUON_AVAILABLE:
    try:
        # Create a simple tensor and test Muon
        test_tensor = torch.randn(10, requires_grad=True)
        muon_optimizer = Muon([test_tensor], lr=0.01)
        print("✓ Muon optimizer created successfully")
        
        # Test a simple optimization step
        loss = test_tensor.sum()
        loss.backward()
        muon_optimizer.step()
        print("✓ Muon optimization step completed successfully")
        
    except Exception as e:
        print(f"✗ Muon functionality test failed: {e}")
        MUON_AVAILABLE = False

✓ Muon imported successfully
✗ Muon functionality test failed: 


In [3]:
# Import our custom modules
try:
    from analysis_scripts.default_paths import WORDNET_ID_TO_HUMAN_PATH
    from analysis_scripts.helpers_16_choice import force_16_choice
    from analysis_scripts.input_helpers import generate_import_image_functions
    from robustness import custom_synthesis_losses
    from robustness.model_utils import make_and_restore_model
    from robustness.tools.distance_measures import *
    from robustness.tools.label_maps import CLASS_DICT
    print("✓ All custom modules imported successfully")
except Exception as e:
    print(f"✗ Failed to import custom modules: {e}")
    raise

In [4]:
# Import the build_network module
import importlib.util

model_dir = 'model_analysis_folders/visual_networks/resnet50'
build_network_path = os.path.join(project_root, model_dir, 'build_network.py')

try:
    # Import build_network module
    build_network_spec = importlib.util.spec_from_file_location(
        "build_network", build_network_path
    )
    build_network = importlib.util.module_from_spec(build_network_spec)
    build_network_spec.loader.exec_module(build_network)
    
    # Load the model
    model, ds, metamer_layers = build_network.main(return_metamer_layers=True)
    print(f"✓ Model loaded successfully")
    print(f"  Model type: {type(model)}")
    print(f"  Metamer layers: {metamer_layers}")
    
except Exception as e:
    print(f"✗ Failed to load model: {e}")
    raise

In [5]:
# Load test image using the same functions as the metamer script
SIDX = 0  # Test with first image
INPUTIMAGEFUNCNAME = "400_16_class_imagenet_val"

try:
    # Generate import image function
    INPUTIMAGEFUNC = generate_import_image_functions(
        INPUTIMAGEFUNCNAME, data_format="NCHW"
    )
    
    # Load image
    image_dict = INPUTIMAGEFUNC(SIDX)
    image_dict["image_orig"] = image_dict["image"]
    
    # Preprocess image (same as metamer script)
    def preproc_image(image, image_dict):
        """The image into the pytorch model should be between 0-1"""
        if image_dict["max_value_image_set"] == 255:
            image = image / 255.0
        return image
    
    image_dict["image"] = preproc_image(image_dict["image"], image_dict)
    
    # Add batch dimension
    im = torch.tensor(np.expand_dims(image_dict["image"], 0)).float().contiguous()
    
    print(f"✓ Image loaded successfully")
    print(f"  Image shape: {im.shape}")
    print(f"  Image range: [{im.min():.3f}, {im.max():.3f}]")
    print(f"  Image filename: {image_dict.get('filename_short', 'unknown')}")
    
except Exception as e:
    print(f"✗ Failed to load image: {e}")
    raise

In [6]:
# Move model to GPU and get original activations
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

model = model.cuda()
model.eval()

# Get original activations
with torch.no_grad():
    (predictions, rep, all_outputs), orig_im = model(
        im.cuda(), with_latent=True, fake_relu=True
    )

print(f"✓ Original activations computed")
print(f"  Available layers: {list(all_outputs.keys())}")
print(f"  Metamer layers: {metamer_layers}")

In [7]:
# Test our modified attacker with Muon optimizer
# We'll test with a single layer first
layer_to_invert = metamer_layers[0]  # Use first layer
print(f"Testing with layer: {layer_to_invert}")

# Set up synthesis parameters (same as metamer script)
synth_kwargs = {
    "custom_loss": custom_synthesis_losses.LOSSES["inversion_loss_layer"](
        layer_to_invert, normalize_loss=True
    ),
    "constraint": "2",
    "eps": 100000,
    "step_size": 1.0,  # This will be learning rate for Muon
    "iterations": 100,  # Reduced for testing
    "do_tqdm": False,
    "targeted": True,
    "use_best": False,
    "optimizer": "muon" if MUON_AVAILABLE else "sgd",
}

print(f"Synthesis parameters:")
for key, value in synth_kwargs.items():
    print(f"  {key}: {value}")

In [8]:
# Prepare input for metamer generation
# Use same noise initialization as metamer script
NOISE_SCALE = 1/20
init_noise_mean = 0.5

# Initialize noise input
im_n_initialized = (
    (torch.randn_like(im) * NOISE_SCALE + init_noise_mean).detach().cpu().numpy()
)

# Clamp to dataset range
im_n = torch.clamp(
    torch.from_numpy(im_n_initialized), ds.min_value, ds.max_value
).cuda()

# Get target representation
invert_rep = (
    all_outputs[layer_to_invert]
    .contiguous()
    .view(all_outputs[layer_to_invert].size(0), -1)
)

print(f"✓ Input prepared for metamer generation")
print(f"  Noise input shape: {im_n.shape}")
print(f"  Target representation shape: {invert_rep.shape}")
print(f"  Noise input range: [{im_n.min():.3f}, {im_n.max():.3f}]")

In [9]:
# Test the actual metamer generation with our modified attacker
print(f"Testing metamer generation with {synth_kwargs['optimizer']} optimizer...")

try:
    # This should use our modified attacker with Muon/SGD support
    (predictions_out, rep_out, all_outputs_out), xadv = model(
        im_n,
        invert_rep.clone(),
        make_adv=True,
        **synth_kwargs,
        with_latent=True,
        fake_relu=True,
    )
    
    print(f"✓ Metamer generation completed successfully!")
    print(f"  Output shape: {xadv.shape}")
    print(f"  Output range: [{xadv.min():.3f}, {xadv.max():.3f}]")
    
    # Compute final loss
    def calc_loss(model, inp, target, custom_loss, should_preproc=True):
        """Calculate loss (same as metamer script)"""
        if should_preproc:
            inp = model.preproc(inp)
        return custom_loss(model.model, inp, target)
    
    final_loss, _ = calc_loss(
        model, xadv, invert_rep.clone(), synth_kwargs["custom_loss"]
    )
    print(f"  Final loss: {final_loss.item():.6f}")
    
except Exception as e:
    print(f"✗ Metamer generation failed: {e}")
    import traceback
    traceback.print_exc()
    raise

In [10]:
# Test both optimizers and compare performance
optimizers_to_test = ["sgd"]
if MUON_AVAILABLE:
    optimizers_to_test.append("muon")

results = {}

for optimizer_name in optimizers_to_test:
    print(f"\nTesting {optimizer_name.upper()} optimizer...")
    
    # Reset synthesis kwargs
    synth_kwargs["optimizer"] = optimizer_name
    
    # Reset input
    im_n = torch.clamp(
        torch.from_numpy(im_n_initialized), ds.min_value, ds.max_value
    ).cuda()
    
    try:
        # Time the optimization
        import time
        start_time = time.time()
        
        (predictions_out, rep_out, all_outputs_out), xadv = model(
            im_n,
            invert_rep.clone(),
            make_adv=True,
            **synth_kwargs,
            with_latent=True,
            fake_relu=True,
        )
        
        end_time = time.time()
        
        # Compute final loss
        final_loss, _ = calc_loss(
            model, xadv, invert_rep.clone(), synth_kwargs["custom_loss"]
        )
        
        results[optimizer_name] = {
            "xadv": xadv,
            "final_loss": final_loss.item(),
            "time": end_time - start_time,
            "predictions_out": predictions_out,
            "all_outputs_out": all_outputs_out
        }
        
        print(f"  ✓ Completed in {results[optimizer_name]['time']:.2f}s")
        print(f"  Final loss: {results[optimizer_name]['final_loss']:.6f}")
        
    except Exception as e:
        print(f"  ✗ Failed: {e}")
        results[optimizer_name] = {"error": str(e)}

In [11]:
# Visualize results
n_optimizers = len(results)
fig, axes = plt.subplots(1, n_optimizers + 1, figsize=(4 * (n_optimizers + 1), 4))

# Original image
orig_img = im.squeeze().cpu().numpy()
if orig_img.shape[0] == 3:  # NCHW format
    orig_img = np.rollaxis(orig_img, 0, 3)
axes[0].imshow(np.clip(orig_img, 0, 1))
axes[0].set_title(f"Original Image")
axes[0].axis('off')

# Results for each optimizer
for i, (optimizer_name, result) in enumerate(results.items()):
    if "error" in result:
        axes[i + 1].text(0.5, 0.5, f"Error:\n{result['error']}", 
                         ha='center', va='center', transform=axes[i + 1].transAxes)
        axes[i + 1].set_title(f"{optimizer_name.upper()} (Failed)")
    else:
        xadv_img = result["xadv"].squeeze().cpu().numpy()
        if xadv_img.shape[0] == 3:  # NCHW format
            xadv_img = np.rollaxis(xadv_img, 0, 3)
        axes[i + 1].imshow(np.clip(xadv_img, 0, 1))
        axes[i + 1].set_title(f"{optimizer_name.upper()}\nLoss: {result['final_loss']:.4f}\nTime: {result['time']:.1f}s")
    axes[i + 1].axis('off')

plt.tight_layout()
plt.show()

# Print comparison summary
print("\nOptimization Results Summary:")
print("=" * 50)
for optimizer_name, result in results.items():
    if "error" in result:
        print(f"{optimizer_name.upper()}: FAILED - {result['error']}")
    else:
        print(f"{optimizer_name.upper()}: Loss={result['final_loss']:.6f}, Time={result['time']:.2f}s")

# Compare if both succeeded
if len(results) > 1 and all("error" not in result for result in results.values()):
    optimizers = list(results.keys())
    loss_diff = results[optimizers[1]]['final_loss'] - results[optimizers[0]]['final_loss']
    time_diff = results[optimizers[1]]['time'] - results[optimizers[0]]['time']
    
    print(f"\nComparison:")
    print(f"  Loss difference ({optimizers[1]} - {optimizers[0]}): {loss_diff:+.6f}")
    print(f"  Time difference ({optimizers[1]} - {optimizers[0]}): {time_diff:+.2f}s")

In [12]:
# Print debug information for porting fixes back to main files
print("\nDebug Information for Main Files:")
print("=" * 50)
print(f"Muon available: {MUON_AVAILABLE}")
print(f"Device used: {device}")
print(f"Model type: {type(model)}")
print(f"Layer tested: {layer_to_invert}")
print(f"Synthesis kwargs used:")
for key, value in synth_kwargs.items():
    print(f"  {key}: {value}")

# Check for any issues that need to be fixed in main files
print("\nIssues to check in main files:")
print("=" * 50)

# Check if Muon import works in main environment
if not MUON_AVAILABLE:
    print("1. Muon not available - need to install or handle fallback")

# Check if any errors occurred
for optimizer_name, result in results.items():
    if "error" in result:
        print(f"2. {optimizer_name.upper()} failed: {result['error']}")
        print(f"   Need to debug this in robustness/attacker.py")

# Check performance
if len(results) > 1 and all("error" not in result for result in results.values()):
    optimizers = list(results.keys())
    if results[optimizers[1]]['final_loss'] > results[optimizers[0]]['final_loss']:
        print(f"3. {optimizers[1].upper()} performed worse than {optimizers[0].upper()}")
        print(f"   May need to tune learning rate or other parameters")

print("\nIf all tests pass, the Muon integration is working correctly!")
print("You can now use the -O muon flag in the main metamer generation script.")